# Tutorial 4 — Retraining ecCount

ecCount learns a probability surface: every gold-standard object becomes a Gaussian
(sigma = 1 px at the network's input resolution), overlapping Gaussians are combined
by taking the maximum, and the network is trained with a weighted binary
cross-entropy plus soft Dice loss against that surface. Counts are read off the
local maxima.

This notebook shows the target the network learns, builds a custom split (a
leave-one-cell-line-out split as the example), writes a self-contained run folder,
and starts training with the same command-line tool that trained the released model.

**Hardware.** The published model was trained for 70 epochs on one GPU (best epoch 49),
at about 80 s per epoch (`train_history.csv`). The short CPU run below only checks
that everything is wired correctly.

In [ ]:
import os, sys, time
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "configs" / "default.yaml").exists():
            return p
    raise RuntimeError("Run this notebook from inside the ecdna-bench repository.")

REPO = find_repo_root()
# Where you downloaded the BioImage Archive files (the folder that contains images/).
DATA_ROOT = Path(os.environ.get("ECDNA_DATA_ROOT", Path.home() / "ecdna_data")).expanduser()
if (DATA_ROOT / "Files" / "images").is_dir():
    DATA_ROOT = DATA_ROOT / "Files"
HAVE_DATA = (DATA_ROOT / "images" / "gt_image").is_dir()
print("repository :", REPO)
print("data folder:", DATA_ROOT, "(found)" if HAVE_DATA else "(not found: the notebook runs on a small synthetic example)")

try:
    import ecdna_bench
    print("ecdna_bench:", Path(ecdna_bench.__file__).parent)
except ImportError:
    sys.path.insert(0, str(REPO / "src"))
    import ecdna_bench
    print("ecdna_bench imported from", REPO / "src")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

NATIVE_SHAPE = (2048, 2448)
ANCHOR_UID = "ncih2170_facs_fish_0723_low_her2_52"   # the example image used in the paper

SUFFIXES = ("", "_pred_roi", "_predicted_roi", "_pred", "_roi", "_mask")

def find_file(folder, uid):
    """The file in `folder` named after the unique identifier (any image extension)."""
    folder = Path(folder) if folder else None
    if folder is None or not folder.is_dir():
        return None
    for suffix in SUFFIXES:
        hits = sorted(p for p in folder.rglob(uid + suffix + ".*")
                      if p.stem == uid + suffix
                      and p.suffix.lower() in {".tif", ".tiff", ".png", ".npy", ".npz"})
        if hits:
            return hits[0]
    return None

def read_image(path, color=False):
    flag = cv2.IMREAD_COLOR if color else cv2.IMREAD_UNCHANGED
    img = cv2.imread(str(path), flag)
    if img is None:
        import tifffile
        img = tifffile.imread(str(path))
    if color:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else np.dstack([img] * 3)
    elif img.ndim == 3:
        img = img.max(axis=2)
    return img

def render_diamonds(points_rc, shape, radius=2):
    """Render (row, col) points as diamonds (|dy| + |dx| <= radius; 13 px for radius 2)."""
    mask = np.zeros(shape, np.uint8)
    offsets = [(dy, dx) for dy in range(-radius, radius + 1)
               for dx in range(-radius, radius + 1) if abs(dy) + abs(dx) <= radius]
    for r, c in np.asarray(points_rc, dtype=int):
        for dy, dx in offsets:
            y, x = r + dy, c + dx
            if 0 <= y < shape[0] and 0 <= x < shape[1]:
                mask[y, x] = 255
    return mask

def synthetic_example(seed=0, n=60):
    """A small stand-in image set, used only when the data folder is missing."""
    rng = np.random.default_rng(seed)
    pts = np.stack([rng.integers(800, 1250, n), rng.integers(1000, 1450, n)], 1)
    gs = render_diamonds(pts, NATIVE_SHAPE)
    rgb = np.full(NATIVE_SHAPE + (3,), 8, np.uint8)
    for r, c in pts:
        cv2.circle(rgb, (int(c), int(r)), 2, (40, 220, 60), -1)
    roi = np.zeros(NATIVE_SHAPE, np.uint8); roi[700:1350, 900:1550] = 255
    return {"uid": "synthetic_example", "rgb": rgb, "dapi": rgb[:, :, 2].copy(),
            "gs": gs, "roi": roi, "points": pts}

def load_image_set(uid=None):
    """Load RGB, DAPI, gold-standard mask, points and ROI for one image set."""
    if not HAVE_DATA:
        return synthetic_example()
    img_dir = DATA_ROOT / "images"
    if uid is None:
        uid = ANCHOR_UID if find_file(img_dir / "gt_image", ANCHOR_UID) else \
            sorted(p.stem for p in (img_dir / "gt_image").iterdir())[0]
    out = {"uid": uid}
    for key, sub, color in [("rgb", "rgb", True), ("dapi", "dapi", False),
                            ("gs", "gt_image", False), ("roi", "roi_mask", False)]:
        p = find_file(img_dir / sub, uid)
        out[key] = read_image(p, color=color) if p else None
    p = find_file(img_dir / "gt_coords", uid)
    out["points"] = np.load(p, allow_pickle=True) if p else None
    return out

## 1. The training target

In [ ]:
from ecdna_bench.eccount.targets import SoftTargetConfig, make_centroid_gaussian_target

sample = load_image_set()
small = cv2.resize((sample["gs"] > 0).astype(np.uint8), (1224, 1024), interpolation=cv2.INTER_NEAREST)
target = make_centroid_gaussian_target(small, SoftTargetConfig(sigma=1.0))
ys, xs = np.nonzero(small)
y0, x0 = ys.min(), xs.min()
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(small[y0:y0 + 120, x0:x0 + 120], cmap="gray"); axes[0].set_title("gold standard at input size")
axes[1].imshow(target[y0:y0 + 120, x0:x0 + 120], cmap="magma"); axes[1].set_title("Gaussian target (sigma = 1 px)")
for ax in axes:
    ax.axis("off")
print("target range:", float(target.min()), "to", float(target.max()))

## 2. A run folder with your own split

`scripts/prepare_local_run.py bia` writes a run folder whose `manifest_local.csv`
lists every image with its `split` column (`train`, `val`, `test`). Training uses the
`train` rows, selects the checkpoint on the `val` rows and never reads `test`.
Changing that column changes the experiment; nothing else needs editing.

Download the training and validation images first
(`docs/TUTORIAL_EXTERNAL.md`, section 6), then run:

In [ ]:
import subprocess, pandas as pd, yaml

BASE_RUN = REPO / "runs" / "retrain"
if HAVE_DATA and not (BASE_RUN / "manifest_local.csv").exists():
    cmd = [sys.executable, str(REPO / "scripts" / "prepare_local_run.py"), "bia",
           "--bia-root", str(DATA_ROOT), "--out-dir", str(BASE_RUN), "--split", "all", "--own-eccount"]
    print(subprocess.run(cmd, cwd=REPO, capture_output=True, text=True).stdout[-1500:])
have_run = (BASE_RUN / "manifest_local.csv").exists()
print("run folder ready:", have_run)

In [ ]:
HOLD_OUT = "SNU16"     # the cell line the model must never see

if have_run:
    m = pd.read_csv(BASE_RUN / "manifest_local.csv")
    loco = m.copy()
    loco.loc[loco.cell_line == HOLD_OUT, "split"] = "test"      # held-out line: evaluation only
    print(loco.groupby(["cell_line", "split"]).size().unstack(fill_value=0))
    assert not ((loco.cell_line == HOLD_OUT) & loco.split.isin(["train", "val"])).any()

    LOCO_RUN = REPO / "runs" / f"loco_{HOLD_OUT.lower()}"
    LOCO_RUN.mkdir(parents=True, exist_ok=True)
    loco.to_csv(LOCO_RUN / "manifest_local.csv", index=False)
    cfg = yaml.safe_load(open(BASE_RUN / "run_config.yaml"))
    cfg["paths"]["consistency_csv"] = str(LOCO_RUN / "manifest_local.csv")
    cfg["paths"]["metadata_csv"] = str(LOCO_RUN / "manifest_local.csv")
    cfg["paths"]["eccount_out_dir"] = str(LOCO_RUN / "eccount_training")
    cfg["paths"]["frozen_results_dir"] = str(LOCO_RUN / "results")
    for key in ("eccount_peaks_masks", "eccount_threshold_masks"):
        cfg["paths"][key] = str(LOCO_RUN / "own_eccount" / key.replace("_masks", ""))
    assert not (LOCO_RUN / "paths.local.yaml").exists(), "a paths.local.yaml here would override this config"
    with open(LOCO_RUN / "run_config.yaml", "w") as fh:
        yaml.safe_dump(cfg, fh, sort_keys=False)
    print("written:", LOCO_RUN / "run_config.yaml")

The printed table is the evidence for the claim that the model never saw the held-out
line: its `train` and `val` counts must be zero.

For the full leave-one-cell-line-out experiment reported in the paper, including the
size-matched control, use `scripts/train_eccount_loco.py`, which adds the same guards
and writes one run folder per held-out line.

## 3. Train

The command below is what you would run in a terminal. `SMOKE_TEST = True` shrinks
the run to eight training and four validation images, one epoch and a small input
size, so that it finishes in minutes on a CPU; it writes to `eccount_training_smoke/`
and the resulting weights are not useful. On a GPU machine set `SMOKE_TEST = False`:
the run then uses every `train` and `val` row, and the device comes from
`eccount.train.device` in the configuration (`cuda`).

In [ ]:
SMOKE_TEST = True

if have_run:
    run_cfg = LOCO_RUN / "run_config.yaml"
    TRAIN_OUT = LOCO_RUN / "eccount_training"
    if SMOKE_TEST:
        # a handful of train and val rows only; the held-out line stays out
        ok = loco[loco["count_mask_consistent"].fillna(False)] if "count_mask_consistent" in loco else loco
        small = pd.concat([ok[ok.split == "train"].head(8), ok[ok.split == "val"].head(4)])
        small.to_csv(LOCO_RUN / "manifest_smoke.csv", index=False)
        TRAIN_OUT = LOCO_RUN / "eccount_training_smoke"
        cfg = yaml.safe_load(open(run_cfg))
        cfg["paths"]["consistency_csv"] = str(LOCO_RUN / "manifest_smoke.csv")
        cfg["paths"]["metadata_csv"] = str(LOCO_RUN / "manifest_smoke.csv")
        cfg["paths"]["eccount_out_dir"] = str(TRAIN_OUT)
        cfg["eccount"]["train"].update(epochs=1, num_workers=0, device="cpu")
        cfg["eccount"]["input_size"] = [256, 304]
        run_cfg = LOCO_RUN / "run_config_smoke.yaml"
        with open(run_cfg, "w") as fh:
            yaml.safe_dump(cfg, fh, sort_keys=False)
        print(small.groupby(["cell_line", "split"]).size().unstack(fill_value=0))
    cmd = [sys.executable, "-m", "ecdna_bench.cli.train_eccount", "--config", str(run_cfg)]
    print("command:", " ".join(cmd))
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True,
                          env={**os.environ, "PYTHONPATH": str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")})
    print(proc.stderr[-2000:])
    print(f"finished in {time.time() - t0:.0f} s")

In [ ]:
if have_run:
    hist = TRAIN_OUT / "train_history.csv"
    if hist.exists():
        h = pd.read_csv(hist)
        display(h.tail())
        if len(h) > 1:
            h.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(6, 3))

## 4. Use the new weights

Point inference at the new checkpoint and score it on the held-out images:

```bash
python scripts/prepare_local_run.py bia --bia-root ~/ecdna_data --out-dir runs/loco_snu16_eval \
    --split all --cell-line SNU16 --own-eccount \
    --checkpoint runs/loco_snu16/eccount_training/best_model.pt
python -m ecdna_bench.cli.run_eccount --config runs/loco_snu16_eval/run_config.yaml --split all
python -m ecdna_bench.cli.benchmark --config runs/loco_snu16_eval/run_config.yaml \
    --models eccount_peaks eccount_mask --skip-harmonize --output-dir runs/loco_snu16_eval/results
```

`summary_by_cell_line.csv` in the results folder gives the held-out line's F1.